# 05 – Ensemble Learning, Neural Network and Final Model Choice
Module owner: Lou-Ann Deley

In this notebook, we try to improve on notebook 03 in two ways. First, we combine several models (ensemble learning). Second, we test a model outside the course: a neural network (multi-layer perceptron). We then compare all models with the baseline using a statistical test, and we choose the final model. We only use the training set: the final model is chosen here, before looking at the test set.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)
from sklearn.neural_network import MLPClassifier

SEED = 42
np.random.seed(SEED)
DATA_DIR = Path("../data")
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
X_train, y_train = train.drop(columns="class"), train["class"]
print("Train:", X_train.shape)

## 1. Models from notebook 03 (best hyperparameters)

In [ ]:
def logreg():
    return LogisticRegression(C=10, max_iter=1000, random_state=SEED)
def rf():
    return RandomForestClassifier(n_estimators=300, max_depth=10, max_features="sqrt",
                                  min_samples_leaf=1, random_state=SEED, n_jobs=-1)
def gb():
    return HistGradientBoostingClassifier(learning_rate=0.1, max_iter=200, max_depth=6,
                                          l2_regularization=0.0, random_state=SEED)

## 2. Ensemble learning
We combine the logistic regression, the Random Forest and the Gradient Boosting in two ways:
- Soft voting: we average the probabilities of the three models.
- Stacking: a logistic regression learns how to best combine the three models, using predictions on data they did not see during training.

In [ ]:
estimators = [("lr", logreg()), ("rf", rf()), ("gb", gb())]
voting = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
stacking = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(max_iter=1000),
                              cv=5, stack_method="predict_proba", n_jobs=-1)

## 3. Model outside the course: a neural network (MLP)
A multi-layer perceptron (MLP) is a small neural network made of layers of connected neurons. Our reference paper is Grinsztajn, Oyallon and Varoquaux (2022), "Why do tree-based models still outperform deep learning on typical tabular data?", NeurIPS 2022. It shows that on tabular data like ours, tree models are usually better than neural networks. We test if this is also true here. We tune a few hyperparameters with cross-validation, scale the data inside the Pipeline, and stop the training early to limit overfitting.

In [ ]:
t0 = time.time()
mlp_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()),
              ("clf", MLPClassifier(max_iter=500, early_stopping=True, random_state=SEED))]),
    {"clf__hidden_layer_sizes": [(32,), (64,), (64, 32)], "clf__alpha": [1e-4, 1e-3, 1e-2]},
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc", n_jobs=-1)
mlp_search.fit(X_train, y_train)
print("Best MLP:", mlp_search.best_params_, "| CV ROC-AUC:", round(mlp_search.best_score_, 4),
      f"| {time.time() - t0:.0f}s")
mlp = mlp_search.best_estimator_

## 4. Comparison of all models with repeated cross-validation
We repeat the 5-fold cross-validation 3 times, which gives 15 scores per model. All models use the same folds, so we can compare them fold by fold.

In [ ]:
candidates = {
    "Logistic regression (baseline)": logreg(),
    "Random Forest": rf(),
    "Gradient Boosting": gb(),
    "Soft voting": voting,
    "Stacking": stacking,
    "MLP (neural network)": mlp,
}
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

scores, times = {}, {}
for name, model in candidates.items():
    t0 = time.time()
    r = cross_validate(model, X_train, y_train, cv=rcv, scoring=["roc_auc", "f1"], n_jobs=1)
    scores[name] = r
    times[name] = time.time() - t0
    print(f"{name:32s} ROC-AUC {r['test_roc_auc'].mean():.4f} ± {r['test_roc_auc'].std():.4f} | "
          f"F1 {r['test_f1'].mean():.4f} | {times[name]:.0f}s")

## 5. Statistical test against the baseline
The 15 scores are not independent, because the training sets overlap. A classic t-test would be too optimistic, so we use the corrected t-test of Nadeau and Bengio (2003). We also give the Wilcoxon test, which does not assume normal data. A p-value below 0.05 means the difference with the baseline is not due to chance.

In [ ]:
def corrected_ttest(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)
    k = len(d)
    var = d.var(ddof=1)
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * var)
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return d.mean(), t, p

n_test_fold = len(X_train) // 5
n_train_fold = len(X_train) - n_test_fold
base = scores["Logistic regression (baseline)"]["test_roc_auc"]

rows = {}
for name, r in scores.items():
    a = r["test_roc_auc"]
    row = {"ROC-AUC": f"{a.mean():.4f} ± {a.std():.4f}", "F1": round(r["test_f1"].mean(), 4),
           "time (s)": round(times[name])}
    if name != "Logistic regression (baseline)":
        diff, t, p = corrected_ttest(a, base, n_train_fold, n_test_fold)
        row.update({"diff vs baseline": round(diff, 4), "p (corrected t-test)": f"{p:.1e}",
                    "p (Wilcoxon)": f"{stats.wilcoxon(a, base).pvalue:.1e}"})
    rows[name] = row
comparison = pd.DataFrame(rows).T
comparison

In [ ]:
mean_auc = pd.Series({n: r["test_roc_auc"].mean() for n, r in scores.items()}).sort_values()
std_auc = pd.Series({n: r["test_roc_auc"].std() for n, r in scores.items()})[mean_auc.index]
ax = mean_auc.plot.barh(xerr=std_auc, color=["tab:gray" if "baseline" in n else "tab:green" for n in mean_auc.index])
ax.set_xlim(mean_auc.min() - 0.01, 1.0)
ax.set_xlabel("ROC-AUC (5-fold CV × 3 repeats, mean ± std)")
ax.set_title("All models vs baseline (grey)")
plt.tight_layout(); plt.savefig(FIG_DIR / "ensembles_mlp_comparison.png", dpi=150); plt.show()

## 6. Choice of the final model
We compare Gradient Boosting with the other good models using the same test. If a more complex model is not significantly better, we keep the simpler one, because it is faster and easier to explain.

In [ ]:
gb_auc = scores["Gradient Boosting"]["test_roc_auc"]
for name in ["Soft voting", "Stacking", "Random Forest", "MLP (neural network)"]:
    diff, t, p = corrected_ttest(scores[name]["test_roc_auc"], gb_auc, n_train_fold, n_test_fold)
    print(f"{name:22s} vs Gradient Boosting: diff = {diff:+.4f}, p = {p:.3f}")

## Conclusions
- All models are significantly better than the baseline (p < 0.01 with both tests).
- Gradient Boosting has the highest ROC-AUC (0.993). Stacking is very close (0.992) and not significantly different (p ≈ 0.12), but it is 8 times slower and harder to explain. Soft voting and Random Forest are significantly below Gradient Boosting.
- The neural network (MLP) is better than the baseline but clearly below the tree models (0.982 vs 0.993). This matches the result of Grinsztajn et al. (2022): on tabular data, tree ensembles usually beat neural networks. Its score is even a bit optimistic, because its hyperparameters were tuned on the same training set.
- Final model: Gradient Boosting with all 30 features. It is chosen here, using cross-validation only, before the test set is used in notebook 06.